# Ep_ISA: Fi-NeMo → ISA → TF Cooperativity + EP-Motif Analysis

- **Genome**: dm3
- **Models**: DeepCAGE (single-task) + DeepSTARR (dual-task: DEV=0, HK=1)
- **Input**: MC000 Stage 2 hits (CAGE/HK/DEV) on 19777 core-promoter regions
- **EP-Motif**: HK/DEV ∩ CAGE positional overlap motifs (enhancer-promoter shared)

In [ ]:
!pip install tensorflow tf-keras bioframe pyBigWig loguru statsmodels h5py matplotlib-venn -q

In [ ]:
import sys, os, json
import pandas as pd, numpy as np
import tensorflow as tf
import h5py, bioframe as bf
from pathlib import Path
from loguru import logger
import matplotlib.pyplot as plt, seaborn as sns
import matplotlib as mpl
mpl.rcParams.update({'font.family':'Arial','font.size':7,'axes.linewidth':0.5,
    'xtick.major.width':0.5,'ytick.major.width':0.5,'xtick.major.size':3,'ytick.major.size':3,
    'savefig.dpi':300,'savefig.bbox':'tight','figure.dpi':150})

from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/JoneSu1/EP_ISA.git /content/EP_ISA 2>/dev/null || true
EP_ISA_SRC = '/content/EP_ISA/src'
if EP_ISA_SRC not in sys.path:
    sys.path.insert(0, EP_ISA_SRC)
from Ep_ISA.quickstart import EpQuickStart

BASE_DIR    = '/content/drive/MyDrive/DeepEpromote/Drosophila'
IC_TRIM_DIR = f'{BASE_DIR}/Motif_cluster/ic_trimmed_results'
SCAN_DIR    = Path(f'{IC_TRIM_DIR}/finemo_validation_scans/stage2_core_promoter/finemo_scans')
FINEMO_H5   = f'{IC_TRIM_DIR}/IC_Trimmed_MetaClusters_for_Finemo.h5'
PROM_FILE   = f'{BASE_DIR}/DeepCAGE/DATA/starr_cage_reconstructed.tsv'
CAGE_MODEL  = f'{BASE_DIR}/DeepCAGE/model/model_starr_ctss_T1/best_model.h5'
STARR_JSON  = f'{BASE_DIR}/DeepSTARR/model_artifacts/DeepSTARR.model.json'
STARR_H5    = f'{BASE_DIR}/DeepSTARR/model_artifacts/DeepSTARR.model.h5'
logger.info('Paths configured.')

## §1 Load Models

In [ ]:
cage_model = tf.keras.models.load_model(
    CAGE_MODEL, custom_objects={'mse': tf.keras.losses.MeanSquaredError()})
print(f'CAGE: in={cage_model.input_shape}, out={cage_model.output_shape}')

import tf_keras
with open(STARR_JSON, 'r') as f:
    src = tf_keras.models.model_from_json(f.read())
src.load_weights(STARR_H5)
tmp = '/content/_tmp_starr.h5'; src.save(tmp)
starr_model = tf.keras.models.load_model(tmp, compile=False); os.remove(tmp)
print(f'DeepSTARR: in={starr_model.input_shape}, out={starr_model.output_shape}  [0]=DEV [1]=HK')

## §2 df_regions (dm3)

In [ ]:
prom = pd.read_csv(PROM_FILE, sep='\t').rename(columns={
    'Chromosome':'chrom','Start':'start','End':'end',
    'CAGE_dom_strand':'strand','CAGE_dom_log2TPM':'y','Sequence':'seq'})
prom['sc'] = 'mismatch'
prom.loc[prom['refseq_tss_strand']==prom['strand'],'sc'] = 'match'
prom.loc[prom[['refseq_tss_strand','strand']].isna().any(axis=1),'sc'] = 'NA'
prom_19777 = prom[prom.sc=='match'].reset_index(drop=True)
assert len(prom_19777)==19777
df_regions = prom_19777[['chrom','start','end']].copy()
df_regions['peak_id'] = df_regions.index
df_regions = df_regions[['peak_id','chrom','start','end']]
print(f'{len(df_regions)} regions')

## §3 FASTA from prom_df (dm3)

In [ ]:
FASTA_PATH = '/content/prom_regions_dm3.fa'
with open(FASTA_PATH, 'w') as f:
    for _, r in prom_19777.iterrows():
        f.write(f">{r['chrom']}:{int(r['start'])}-{int(r['end'])}\n{r['seq']}\n")
logger.info(f'Built {FASTA_PATH}')

## §4 Motif Overlap & EP-Motif Definition

In [ ]:
mapping = {}
with h5py.File(FINEMO_H5,'r') as f:
    for pn, g in f['pos_patterns'].items():
        mc=g.attrs.get('mc_id',b''); tf=g.attrs.get('tf_label',b'')
        if isinstance(mc,bytes): mc=mc.decode()
        if isinstance(tf,bytes): tf=tf.decode()
        if not tf: tf='Unknown'
        mapping[f'pos_patterns.{pn}']={'MC_ID':mc,'TF':tf}

def load_hits(task):
    df=pd.read_csv(SCAN_DIR/task/'hits.tsv',sep='\t')
    df['MC_ID']=df['motif_name'].map(lambda x:mapping.get(x,{}).get('MC_ID',x))
    df['TF']=df['motif_name'].map(lambda x:mapping.get(x,{}).get('TF','Unknown'))
    return df

cage=load_hits('CAGE_NEW'); hk=load_hits('HK'); dev=load_hits('DEV')
print(f'CAGE:{len(cage)} HK:{len(hk)} DEV:{len(dev)}')

In [ ]:
# --- EP definition: seq-level co-occurrence ---
# EP = same seq has hits from BOTH CAGE and HK/DEV (regardless of MC_ID)
# EP_H = CAGE + HK co-occur; EP_D = CAGE + DEV co-occur
cage_seqs=set(cage.peak_id.unique()); hk_seqs=set(hk.peak_id.unique()); dev_seqs=set(dev.peak_id.unique())
ep_h_seqs=cage_seqs & hk_seqs
ep_d_seqs=cage_seqs & dev_seqs
ep_seqs=ep_h_seqs|ep_d_seqs
cage_only_seqs=cage_seqs-ep_seqs; hk_only_seqs=hk_seqs-cage_seqs; dev_only_seqs=dev_seqs-cage_seqs
print(f'Seqs: CAGE={len(cage_seqs)} HK={len(hk_seqs)} DEV={len(dev_seqs)}')
print(f'EP_H={len(ep_h_seqs)}  EP_D={len(ep_d_seqs)}  EP_total={len(ep_seqs)}')
print(f'CAGE-only={len(cage_only_seqs)} HK-only={len(hk_only_seqs)} DEV-only={len(dev_only_seqs)}')

# --- Positional overlap within EP seqs ---
# same_pos: CAGE and HK/DEV hits OVERLAP in position on same seq
# diff_pos: same seq has both CAGE and HK/DEV hits but at DIFFERENT positions
def seqs_with_pos_overlap(df_a, df_b):
    m=df_a[['peak_id','start','end']].merge(df_b[['peak_id','start','end']],on='peak_id',suffixes=('_a','_b'))
    if m.empty: return set()
    has_ov=(m[['end_a','end_b']].min(axis=1)-m[['start_a','start_b']].max(axis=1))>0
    return set(m.loc[has_ov,'peak_id'].unique())

ep_h_same=seqs_with_pos_overlap(cage[cage.peak_id.isin(ep_h_seqs)],hk[hk.peak_id.isin(ep_h_seqs)])
ep_h_diff=ep_h_seqs-ep_h_same
ep_d_same=seqs_with_pos_overlap(cage[cage.peak_id.isin(ep_d_seqs)],dev[dev.peak_id.isin(ep_d_seqs)])
ep_d_diff=ep_d_seqs-ep_d_same
print(f'\nEP_H: same_pos={len(ep_h_same)} diff_pos={len(ep_h_diff)}')
print(f'EP_D: same_pos={len(ep_d_same)} diff_pos={len(ep_d_diff)}')

# TF-level: TFs found in EP seqs (for downstream cooperativity comparison)
ep_tf=sorted(set(
    cage[cage.peak_id.isin(ep_seqs)].TF.unique().tolist()+
    hk[hk.peak_id.isin(ep_seqs)].TF.unique().tolist()+
    dev[dev.peak_id.isin(ep_seqs)].TF.unique().tolist()))

In [ ]:
from matplotlib_venn import venn3
fig,axes=plt.subplots(1,2,figsize=(13,5))
venn3([cage_seqs,hk_seqs,dev_seqs],('CAGE','HK','DEV'),ax=axes[0])
axes[0].set_title('Seq Co-occurrence (EP)')
cats=['EP_H\nsame pos','EP_H\ndiff pos','EP_D\nsame pos','EP_D\ndiff pos','CAGE\nonly','HK only','DEV only']
counts=[len(ep_h_same),len(ep_h_diff),len(ep_d_same),len(ep_d_diff),len(cage_only_seqs),len(hk_only_seqs),len(dev_only_seqs)]
colors=['#d62728','#ff9896','#ff7f0e','#ffbb78','#1f77b4','#2ca02c','#9467bd']
axes[1].bar(cats,counts,color=colors)
axes[1].set_ylabel('Number of seqs'); axes[1].set_title('Seq Classification')
for i,c in enumerate(counts): axes[1].text(i,c+5,str(c),ha='center',fontsize=8)
plt.tight_layout(); plt.savefig('motif_classification.png',dpi=150); plt.show()

## §4.5 Motif–TSS Distance Analysis

Download dm3 RefSeq TSS from UCSC, map each promoter to its real TSS offset, then compute motif→TSS distances.

In [ ]:
# --- Download dm3 RefSeq gene annotation from UCSC ---
ref = pd.read_csv(
    'https://hgdownload.soe.ucsc.edu/goldenPath/dm3/database/refGene.txt.gz',
    sep='\t', header=None, compression='gzip',
    usecols=[1,2,3,4,5],
    names=['transcript','chrom','strand','txStart','txEnd'])
ref['tss'] = ref.apply(lambda r: r['txStart'] if r['strand']=='+' else r['txEnd'], axis=1)
print(f'RefSeq dm3: {len(ref)} transcripts, {ref.chrom.nunique()} chroms')

tss_bed = ref[['chrom','tss','tss','transcript','strand']].copy()
tss_bed.columns = ['chrom','start','end','transcript','strand']
tss_bed['end'] = tss_bed['start'] + 1
tss_bed = tss_bed.drop_duplicates(subset=['chrom','start','strand']).reset_index(drop=True)
print(f'Unique TSS positions: {len(tss_bed)}')

In [ ]:
# --- Map each promoter window to its real TSS ---
prom_bed = df_regions[['chrom','start','end','peak_id']].copy()

overlap = bf.overlap(
    prom_bed, tss_bed, on=['chrom'],
    suffixes=('_prom','_tss'), how='inner'
)
valid = overlap[overlap['start_tss'].notna()].copy()
if valid.empty:
    raise ValueError('No TSS overlaps! Check chrom names (dm3 uses chr2L/chr2R/chr3L/chr3R/chrX/chr4).')

valid['tss_pos']    = valid['start_tss'].astype(int)
valid['tss_offset'] = valid['tss_pos'] - valid['start_prom'].astype(int)
valid['dist_to_center'] = (valid['tss_offset'] - 124).abs()
best = valid.loc[valid.groupby('peak_id')['dist_to_center'].idxmin()]
tss_map = best[['peak_id','tss_offset','strand']].rename(
    columns={'strand':'gene_strand'}).reset_index(drop=True)

n_mapped = len(tss_map)
print(f'TSS mapped: {n_mapped} / {len(prom_bed)} promoters ({n_mapped/len(prom_bed)*100:.1f}%)')
print(f'TSS offset: min={tss_map.tss_offset.min()}, max={tss_map.tss_offset.max()}, '
      f'mean={tss_map.tss_offset.mean():.1f}, median={tss_map.tss_offset.median():.0f}')
print(f'(Geometric center = 124. If TSS were fixed, all offsets would be ~124.)')

In [ ]:
# --- Compute motif-center → TSS distance for all hits ---
def compute_tss_dist(hits_df, tss_map):
    df = hits_df.merge(tss_map, on='peak_id', how='left')
    df['motif_center'] = (df['start'] + df['end']) // 2
    df['tss_dist_seq'] = df['motif_center'] - df['tss_offset']
    df['tss_dist_bp'] = df.apply(
        lambda r: r['tss_dist_seq'] if pd.isna(r.get('gene_strand')) or r['gene_strand']=='+'
        else -r['tss_dist_seq'], axis=1)
    return df

cage_tss = compute_tss_dist(cage, tss_map)
hk_tss   = compute_tss_dist(hk, tss_map)
dev_tss  = compute_tss_dist(dev, tss_map)

for name, df in [('CAGE',cage_tss),('HK',hk_tss),('DEV',dev_tss)]:
    has = df['tss_offset'].notna()
    med = df.loc[has,'tss_dist_bp'].abs().median()
    print(f'{name}: {has.sum()}/{len(df)} hits mapped, median |dist|={med:.0f}bp')

In [ ]:
# --- Plot 1: TSS offset distribution (proves TSS is NOT fixed) ---
# --- Plot 2: motif-TSS distance KDE by task ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(tss_map['tss_offset'], bins=50, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].axvline(124, color='red', ls='--', lw=1.5, label='Geometric center (124)')
axes[0].axvline(tss_map.tss_offset.median(), color='orange', ls='-', lw=1.5,
                label=f'Median ({tss_map.tss_offset.median():.0f})')
axes[0].set_xlabel('TSS offset within 249bp window')
axes[0].set_ylabel('Promoter count')
axes[0].set_title('TSS Position is NOT Fixed')
axes[0].legend(fontsize=8)

for name, df, color in [('CAGE',cage_tss,'#1f77b4'),('HK',hk_tss,'#ff7f0e'),('DEV',dev_tss,'#2ca02c')]:
    v = df[df['tss_dist_bp'].notna()]
    if len(v) > 0:
        sns.kdeplot(data=v, x='tss_dist_bp', color=color, label=name, ax=axes[1], alpha=0.4)
axes[1].axvline(0, color='black', ls='--', lw=0.5)
axes[1].set_xlabel('Distance to TSS (bp, negative=upstream)')
axes[1].set_title('Motif–TSS Distance by Task')
axes[1].legend()

plt.tight_layout(); plt.savefig('motif_tss_distance.png', dpi=150); plt.show()

In [ ]:
# --- Plot 3: EP vs task-specific TSS distance (3-panel boxplot) ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, df) in zip(axes, [('CAGE',cage_tss),('DEV',dev_tss),('HK',hk_tss)]):
    v = df[df['tss_dist_bp'].notna()].copy()
    if v.empty:
        ax.set_title(f'{name}: no data'); continue
    v['is_ep'] = v['peak_id'].isin(ep_seqs)
    sns.boxplot(data=v, x='is_ep', y='tss_dist_bp', hue='is_ep', ax=ax,
                palette={True:'#d62728',False:'#1f77b4'}, legend=False)
    ax.set_xticklabels(['Specific','EP']); ax.set_xlabel(''); ax.set_title(name)
    ax.axhline(0, color='grey', ls='--', lw=0.5)
plt.suptitle('Motif–TSS Distance: EP vs Task-Specific', y=1.02)
plt.tight_layout(); plt.savefig('ep_vs_specific_tss_dist.png', dpi=150); plt.show()

In [ ]:
# --- Plot 4: Per-TF median TSS distance heatmap (cross-task) ---
tf_tss = {}
for name, df in [('CAGE',cage_tss),('DEV',dev_tss),('HK',hk_tss)]:
    v = df[df['tss_dist_bp'].notna()]
    if v.empty: continue
    tf_tss[name] = v.groupby('TF')['tss_dist_bp'].median()
tf_tss_df = pd.DataFrame(tf_tss)
shared = tf_tss_df.dropna(thresh=2).sort_values('CAGE')
if not shared.empty:
    fig, ax = plt.subplots(figsize=(4, max(6, len(shared)*0.28)))
    sns.heatmap(shared, cmap='coolwarm', center=0, annot=True, fmt='.0f',
                annot_kws={'size':6}, ax=ax)
    ax.set_title('Median Motif–TSS Distance by TF')
    plt.tight_layout(); plt.savefig('tf_tss_heatmap.png', dpi=150); plt.show()
else:
    print('No shared TFs across tasks for heatmap.')

In [ ]:
# --- Plot 5: TSS distance vs cooperativity score (scatter) ---
fig, ax = plt.subplots(figsize=(5, 4))
for name, rd, track, color in [('CAGE','./results_cage',0,'#1f77b4'),
                                 ('DEV','./results_dev',0,'#ff7f0e'),
                                 ('HK','./results_hk',1,'#2ca02c')]:
    coop_p = f'{rd}/Data/coop_tf_pair_t{track}.csv'
    if not os.path.exists(coop_p): continue
    cp = pd.read_csv(coop_p)
    sig = cp[cp.cooperativity!='Independent'].copy()
    if sig.empty: continue
    # average TSS distance for each TF pair
    tss_lookup = {}
    for src, df_t in [(name,{'CAGE':cage_tss,'DEV':dev_tss,'HK':hk_tss}[name])]:
        v = df_t[df_t.tss_dist_bp.notna()].groupby('TF')['tss_dist_bp'].median()
        tss_lookup[src] = v.to_dict()
    tvals = {**tss_lookup.get(name,{})}
    sig['tss_dist'] = sig['tf_pair'].apply(
        lambda p: np.nanmean([tvals.get(t, np.nan) for t in str(p).split('|')]))
    valid = sig[sig['tss_dist'].notna()]
    if len(valid)>0:
        ax.scatter(valid['tss_dist'], valid['coop_score'], color=color,
                   label=name, alpha=0.5, s=15)
ax.axvline(0, color='grey', ls='--', lw=0.5)
ax.set_xlabel('Avg Motif–TSS Distance (bp)'); ax.set_ylabel('Coop Score')
ax.set_title('TSS Distance vs Cooperativity'); ax.legend()
plt.tight_layout(); plt.savefig('tss_dist_vs_coop.png', dpi=150); plt.show()

In [ ]:
# --- Plot 6: Per-motif TSS distance boxplot (top 10 TFs per task) ---
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
for ax, (name, df_t) in zip(axes, [('CAGE',cage_tss),('DEV',dev_tss),('HK',hk_tss)]):
    v = df_t[df_t['tss_dist_bp'].notna()].copy()
    if v.empty:
        ax.set_title(f'{name}: no data'); continue
    top10 = v.groupby('TF').size().nlargest(10).index.tolist()
    v10 = v[v['TF'].isin(top10)]
    order = v10.groupby('TF')['tss_dist_bp'].median().sort_values().index
    sns.boxplot(data=v10, y='TF', x='tss_dist_bp', order=order, ax=ax,
                color='steelblue', fliersize=2, linewidth=0.5)
    ax.axvline(0, color='red', ls='--', lw=0.5)
    ax.set_xlim(-200, 200)
    ax.set_xlabel('Distance to TSS (bp)', fontsize=7)
    ax.set_ylabel('')
    ax.set_title(name, fontsize=8)
    ax.tick_params(axis='y', labelsize=6)
plt.suptitle('Motif–TSS Distance by TF (Top 10)', fontsize=9)
plt.tight_layout(); plt.savefig('motif_tss_boxplot_top10.png', dpi=300, bbox_inches='tight'); plt.show()

## §5 ISA — CAGE
Full pipeline + 17 plots via `qs.report()`.

In [ ]:
qs_cage=EpQuickStart(results_dir='./results_cage',fasta_path=FASTA_PATH,df_regions=df_regions)
qs_cage.define_model(cage_model)
qs_cage.load_finemo(hits_tsv_path=str(SCAN_DIR/'CAGE_NEW'/'hits.tsv'),finemo_h5_path=FINEMO_H5,auto_threshold_percentile=50)
qs_cage.run_isa(isa_config={'tracks':[0],'null_percentile':80,'min_count':10,'q_val_thresh':0.1})
qs_cage.report()
print('CAGE done → ./results_cage/Plots/')

## §6 ISA — DEV (DeepSTARR track=0)

In [ ]:
qs_dev=EpQuickStart(results_dir='./results_dev',fasta_path=FASTA_PATH,df_regions=df_regions)
qs_dev.define_model(starr_model)
qs_dev.load_finemo(hits_tsv_path=str(SCAN_DIR/'DEV'/'hits.tsv'),finemo_h5_path=FINEMO_H5,auto_threshold_percentile=50)
qs_dev.run_isa(isa_config={'tracks':[0],'null_percentile':80,'min_count':10,'q_val_thresh':0.1})
qs_dev.report()
print('DEV done → ./results_dev/Plots/')

## §7 ISA — HK (DeepSTARR track=1)

In [ ]:
qs_hk=EpQuickStart(results_dir='./results_hk',fasta_path=FASTA_PATH,df_regions=df_regions)
qs_hk.define_model(starr_model)
qs_hk.load_finemo(hits_tsv_path=str(SCAN_DIR/'HK'/'hits.tsv'),finemo_h5_path=FINEMO_H5,auto_threshold_percentile=50)
qs_hk.run_isa(isa_config={'tracks':[1],'null_percentile':80,'min_count':10,'q_val_thresh':0.1})
qs_hk.report()
print('HK done → ./results_hk/Plots/')

## §8 Cross-Task Full Report (17 plots)

Loads all ISA outputs from CAGE/DEV/HK, overlays them on every visualization function.

In [ ]:
# --- Load all ISA data per task ---
def load_isa(rd, track):
    d={}
    for key, fn in [('coop_pair',f'coop_tf_pair_t{track}.csv'),('coop_tf',f'coop_tf_t{track}.csv'),
                     ('imp','tf_importance.csv'),('isa_combi','motif_combi_isa.csv'),
                     ('null_isa','null_isa.csv'),('null_inter','null_interaction.csv')]:
        p=os.path.join(rd,'Data',fn)
        d[key]=pd.read_csv(p) if os.path.exists(p) else pd.DataFrame()
    return d

CAGE_D=load_isa('./results_cage',0)
DEV_D =load_isa('./results_dev',0)
HK_D  =load_isa('./results_hk',1)

TASKS=[('CAGE',CAGE_D,'#1f77b4'),('DEV',DEV_D,'#ff7f0e'),('HK',HK_D,'#2ca02c')]
OUT='./cross_task_plots'
os.makedirs(OUT,exist_ok=True)

ep_set=set(ep_tf)
def has_ep(s): return any(t in ep_set for t in str(s).split('|'))

def sig(df):
    return df[df.cooperativity!='Independent'].copy() if 'cooperativity' in df.columns and not df.empty else pd.DataFrame()

print('Data loaded.')

In [ ]:
# Plot 1: coop_score_hist (3-task overlay)
fig,ax=plt.subplots(figsize=(4,3))
for name,d,color in TASKS:
    s=sig(d['coop_pair'])
    if s.empty: continue
    sns.histplot(s,x='coop_score',color=color,label=name,alpha=0.4,ax=ax,stat='density')
ax.legend(); ax.set_title('Coop Score Distribution')
plt.tight_layout(); plt.savefig(f'{OUT}/01_coop_hist.png',dpi=150); plt.show()

# Plot 2: coop_score by EP vs specific (3-panel)
fig,axes=plt.subplots(1,3,figsize=(12,3.5))
for ax,(name,d,_) in zip(axes,TASKS):
    s=sig(d['coop_pair'])
    if s.empty: ax.set_title(f'{name}: no data'); continue
    s=s.copy(); s['ep']=s['tf_pair'].apply(has_ep)
    sns.boxplot(data=s,x='ep',y='coop_score',hue='ep',ax=ax,palette={True:'#d62728',False:'#1f77b4'},legend=False)
    ax.set_xticklabels(['Specific','EP']); ax.set_title(name); ax.axhline(0,color='grey',ls='--',lw=0.5)
plt.suptitle('EP vs Task-Specific Cooperativity',y=1.02)
plt.tight_layout(); plt.savefig(f'{OUT}/02_ep_vs_specific_coop.png',dpi=150); plt.show()

In [ ]:
# Plot 3: coop_score_heatmap (merged best-score per pair)
merged=[]
for name,d,_ in TASKS:
    s=sig(d['coop_pair'])
    if s.empty: continue
    s=s.copy(); s['task']=name; merged.append(s)
if merged:
    allm=pd.concat(merged)
    pivot=allm.pivot_table(index='tf_pair',columns='task',values='coop_score',aggfunc='first').dropna(thresh=2)
    top=pivot.assign(_mx=pivot.abs().max(axis=1)).sort_values('_mx',ascending=False).head(30).drop('_mx',axis=1)
    fig,ax=plt.subplots(figsize=(5,8))
    sns.heatmap(top,cmap='coolwarm',vmin=-1,vmax=1,center=0,annot=True,fmt='.2f',annot_kws={'size':5},ax=ax)
    ax.set_title('Cross-Task Cooperativity (Top 30)')
    plt.tight_layout(); plt.savefig(f'{OUT}/03_coop_heatmap.png',dpi=150); plt.show()

In [ ]:
# Plot 4: null_isa KDE (3-task overlay)
fig,ax=plt.subplots(figsize=(4,3))
for name,d,color in TASKS:
    if d['null_isa'].empty: continue
    col=[c for c in d['null_isa'].columns if c.startswith('isa_t')]
    if not col: continue
    sns.kdeplot(data=d['null_isa'],x=col[0],color=color,label=name,alpha=0.4,ax=ax)
ax.legend(); ax.set_title('Null ISA Distribution')
plt.tight_layout(); plt.savefig(f'{OUT}/04_null_isa.png',dpi=150); plt.show()

# Plot 5: null_interaction KDE
fig,ax=plt.subplots(figsize=(4,3))
for name,d,color in TASKS:
    if d['null_inter'].empty: continue
    col=[c for c in d['null_inter'].columns if c.startswith('interaction_t')]
    if not col: continue
    sns.kdeplot(data=d['null_inter'],x=col[0],color=color,label=name,alpha=0.4,ax=ax)
ax.legend(); ax.set_title('Null Interaction Distribution')
plt.tight_layout(); plt.savefig(f'{OUT}/05_null_inter.png',dpi=150); plt.show()

In [ ]:
# Plot 6: interaction_decay (3-task overlay)
fig,ax=plt.subplots(figsize=(4,3))
for name,d,color in TASKS:
    if d['isa_combi'].empty: continue
    col=[c for c in d['isa_combi'].columns if c.startswith('interaction_t')]
    if not col: continue
    decay=d['isa_combi'].assign(abs_v=d['isa_combi'][col[0]].abs()).groupby('distance')['abs_v'].mean().reset_index()
    ax.plot(decay['distance'],decay['abs_v'],color=color,label=name,lw=1)
ax.legend(); ax.set_xlabel('Distance (bp)'); ax.set_ylabel('Mean |Interaction|'); ax.set_title('Interaction Decay')
plt.tight_layout(); plt.savefig(f'{OUT}/06_interaction_decay.png',dpi=150); plt.show()

In [ ]:
# Plot 7: distance vs coop_score scatter (3-task color)
fig,ax=plt.subplots(figsize=(4,3))
for name,d,color in TASKS:
    s=sig(d['coop_pair'])
    if s.empty or 'median_distance' not in s.columns: continue
    ax.scatter(s['median_distance'],s['coop_score'],color=color,label=name,alpha=0.5,s=10)
ax.legend(); ax.set_xlabel('Median Distance'); ax.set_ylabel('Coop Score'); ax.set_title('Distance vs Cooperativity')
ax.axhline(0,color='grey',ls='--',lw=0.5)
plt.tight_layout(); plt.savefig(f'{OUT}/07_distance_scatter.png',dpi=150); plt.show()

In [ ]:
# Plot 8: coop_vs_importance (3-task color)
fig,ax=plt.subplots(figsize=(4,3))
for name,d,color in TASKS:
    ct=d['coop_tf']; imp=d['imp']
    if ct.empty or imp.empty: continue
    s_ct=sig(ct) if 'cooperativity' in ct.columns else ct
    isa_col=[c for c in imp.columns if c.startswith('mean_isa')]
    if not isa_col: continue
    m=s_ct[['tf','coop_score']].merge(imp[['tf',isa_col[0]]],on='tf',how='inner')
    ax.scatter(m['coop_score'],m[isa_col[0]],color=color,label=name,alpha=0.5,s=10)
ax.legend(); ax.set_xlabel('Coop Score'); ax.set_ylabel('TF Importance'); ax.set_title('Cooperativity vs Importance')
plt.tight_layout(); plt.savefig(f'{OUT}/08_coop_vs_importance.png',dpi=150); plt.show()

In [ ]:
# Plot 9: EP vs specific TF importance (3-panel)
fig,axes=plt.subplots(1,3,figsize=(12,3.5))
for ax,(name,d,_) in zip(axes,TASKS):
    imp=d['imp']
    if imp.empty: ax.set_title(f'{name}: no data'); continue
    isa_col=[c for c in imp.columns if c.startswith('mean_isa')]
    if not isa_col: ax.set_title(f'{name}: no ISA'); continue
    imp=imp.copy(); imp['ep']=imp['tf'].isin(ep_set)
    sns.boxplot(data=imp,x='ep',y=isa_col[0],hue='ep',ax=ax,palette={True:'#d62728',False:'#1f77b4'},legend=False)
    ax.set_xticklabels(['Specific','EP']); ax.set_title(name)
plt.suptitle('TF Importance: EP vs Specific',y=1.02)
plt.tight_layout(); plt.savefig(f'{OUT}/09_ep_vs_specific_imp.png',dpi=150); plt.show()

In [ ]:
# Plot 10: partner_specificity KDE (3-task)
fig,ax=plt.subplots(figsize=(4,3))
for name,d,color in TASKS:
    cp=d['coop_pair']; ct=d['coop_tf']
    if cp.empty or ct.empty: continue
    s_cp=sig(cp)
    if s_cp.empty or 'abs_i_sum' not in s_cp.columns: continue
    pairs=s_cp['tf_pair'].str.split('|',expand=True)
    df_long=pd.concat([s_cp[['abs_i_sum']].assign(tf=pairs[0]),s_cp[['abs_i_sum']].assign(tf=pairs[1])])
    def gratio(g):
        if len(g)<10: return np.nan
        return g.nlargest(5,'abs_i_sum')['abs_i_sum'].sum()/g['abs_i_sum'].sum()
    ratios=df_long.groupby('tf').apply(gratio).dropna()
    if len(ratios)>0: sns.kdeplot(ratios,color=color,label=name,ax=ax)
ax.legend(); ax.set_xlabel('Top-5 Partner Ratio'); ax.set_title('Partner Specificity')
plt.tight_layout(); plt.savefig(f'{OUT}/10_partner_specificity.png',dpi=150); plt.show()

In [ ]:
# Plot 11: EP-motif cross-task coop heatmap (only pairs containing EP TFs)
if merged:
    ep_pairs=allm[allm['tf_pair'].apply(has_ep)]
    if not ep_pairs.empty:
        piv_ep=ep_pairs.pivot_table(index='tf_pair',columns='task',values='coop_score',aggfunc='first').dropna(thresh=2)
        top_ep=piv_ep.assign(_mx=piv_ep.abs().max(axis=1)).sort_values('_mx',ascending=False).head(25).drop('_mx',axis=1)
        fig,ax=plt.subplots(figsize=(5,7))
        sns.heatmap(top_ep,cmap='coolwarm',vmin=-1,vmax=1,center=0,annot=True,fmt='.2f',annot_kws={'size':5},ax=ax)
        ax.set_title('EP-Motif Pairs: Cross-Task Cooperativity')
        plt.tight_layout(); plt.savefig(f'{OUT}/11_ep_heatmap.png',dpi=150); plt.show()

In [ ]:
# Plot 12: cooperativity category counts (3-task bar)
fig,ax=plt.subplots(figsize=(5,3))
cat_counts={}
for name,d,_ in TASKS:
    cp=d['coop_pair']
    if cp.empty or 'cooperativity' not in cp.columns: continue
    vc=cp[cp.cooperativity!='Independent']['cooperativity'].value_counts()
    cat_counts[name]=vc
if cat_counts:
    cdf=pd.DataFrame(cat_counts).fillna(0)
    cdf.plot(kind='bar',ax=ax,color={'CAGE':'#1f77b4','DEV':'#ff7f0e','HK':'#2ca02c'})
    ax.set_title('Cooperativity Category Counts'); ax.set_ylabel('Pairs')
plt.tight_layout(); plt.savefig(f'{OUT}/12_category_counts.png',dpi=150); plt.show()

In [ ]:
# Plot 13: coop_score ECDF by EP/specific (3-task × 2-category)
fig,axes=plt.subplots(1,3,figsize=(12,3.5))
for ax,(name,d,color) in zip(axes,TASKS):
    s=sig(d['coop_pair'])
    if s.empty: ax.set_title(f'{name}: no data'); continue
    s=s.copy(); s['ep']=s['tf_pair'].apply(has_ep)
    for ep_val,lbl,col in [(True,'EP','#d62728'),(False,'Specific','#1f77b4')]:
        sub=s[s.ep==ep_val]
        if sub.empty: continue
        sns.ecdfplot(sub,x='coop_score',color=col,label=lbl,ax=ax)
    ax.legend(); ax.set_title(name)
plt.suptitle('Coop Score ECDF: EP vs Specific',y=1.02)
plt.tight_layout(); plt.savefig(f'{OUT}/13_ecdf_ep_vs_specific.png',dpi=150); plt.show()

In [ ]:
# Plot 14: n_effective comparison (3-task box)
fig,ax=plt.subplots(figsize=(4,3))
ndata=[]
for name,d,color in TASKS:
    s=sig(d['coop_pair'])
    if s.empty or 'n_effective' not in s.columns: continue
    s=s.copy(); s['task']=name
    ndata.append(s[['task','n_effective']])
if ndata:
    ndf=pd.concat(ndata)
    sns.boxplot(data=ndf,x='task',y='n_effective',hue='task',ax=ax,palette={'CAGE':'#1f77b4','DEV':'#ff7f0e','HK':'#2ca02c'},legend=False)
    ax.set_title('Effective Sample Size')
plt.tight_layout(); plt.savefig(f'{OUT}/14_n_effective.png',dpi=150); plt.show()

In [ ]:
# Plot 15: median_distance comparison (3-task box)
fig,ax=plt.subplots(figsize=(4,3))
ddata=[]
for name,d,color in TASKS:
    s=sig(d['coop_pair'])
    if s.empty or 'median_distance' not in s.columns: continue
    s=s.copy(); s['task']=name
    ddata.append(s[['task','median_distance']])
if ddata:
    ddf=pd.concat(ddata)
    sns.boxplot(data=ddf,x='task',y='median_distance',hue='task',ax=ax,palette={'CAGE':'#1f77b4','DEV':'#ff7f0e','HK':'#2ca02c'},legend=False)
    ax.set_title('Median Motif Distance')
plt.tight_layout(); plt.savefig(f'{OUT}/15_distance.png',dpi=150); plt.show()

In [ ]:
# Plot 16: mw_q (significance) comparison (3-task -log10)
fig,ax=plt.subplots(figsize=(4,3))
qdata=[]
for name,d,color in TASKS:
    s=sig(d['coop_pair'])
    if s.empty or 'mw_q' not in s.columns: continue
    s=s.copy(); s['neglogq']=-np.log10(s['mw_q'].clip(lower=1e-50)); s['task']=name
    qdata.append(s[['task','neglogq']])
if qdata:
    qdf=pd.concat(qdata)
    sns.boxplot(data=qdf,x='task',y='neglogq',hue='task',ax=ax,palette={'CAGE':'#1f77b4','DEV':'#ff7f0e','HK':'#2ca02c'},legend=False)
    ax.set_title('Significance (-log10 q)')
plt.tight_layout(); plt.savefig(f'{OUT}/16_significance.png',dpi=150); plt.show()

In [ ]:
# Plot 17: TF pair count Venn (CAGE/DEV/HK significant pairs)
from matplotlib_venn import venn3
pair_sets={}
for name,d,_ in TASKS:
    s=sig(d['coop_pair'])
    pair_sets[name]=set(s['tf_pair']) if not s.empty else set()
fig,ax=plt.subplots(figsize=(4,4))
venn3([pair_sets.get('CAGE',set()),pair_sets.get('DEV',set()),pair_sets.get('HK',set())],
      ('CAGE','DEV','HK'),ax=ax)
ax.set_title('Significant TF Pair Overlap')
plt.tight_layout(); plt.savefig(f'{OUT}/17_pair_venn.png',dpi=150); plt.show()
print(f'Sig pairs: CAGE={len(pair_sets.get("CAGE",{}))} DEV={len(pair_sets.get("DEV",{}))} HK={len(pair_sets.get("HK",{}))}')

## Output Summary

| Section | Outputs |
|---|---|
| §4 EP-Motif | `motif_classification.png`, `ep_motifs_detail.csv` |
| §4.5 Motif-TSS | `motif_tss_distance.png`, `ep_vs_specific_tss_dist.png`, `tf_tss_heatmap.png`, `tss_dist_vs_coop.png` |
| §5 CAGE ISA | `./results_cage/Plots/` (17 plots) |
| §6 DEV ISA | `./results_dev/Plots/` (17 plots) |
| §7 HK ISA | `./results_hk/Plots/` (17 plots) |
| §8 Cross-Task | `./cross_task_plots/01_*.png` through `17_*.png` (17 plots) |